# Baseline y diseño experimental — Telco Customer Churn (P2)

**Propósito:** evaluar un clasificador de referencia (`DummyClassifier`, clase mayoritaria) con el conjunto **Validation**, y documentar un protocolo común para compararlo con regresión logística y árbol de decisión.

**Ejecución:** seguir las celdas en orden. Se utiliza `preprocessing.py`, compartido con el equipo. El notebook funciona desde la raíz del repositorio o desde `notebooks/`. En Google Colab, si el repositorio ya incluye `preprocessing.py`, se descargará automáticamente cuando sea necesario; de lo contrario, debe subirse el archivo al entorno de Colab antes de ejecutar.

> **Importante:** no se calculan métricas sobre **Test** en este notebook. La comparación final de los tres modelos se hará después de cerrar la selección de hiperparámetros.

## 1. Importación de librerías y localización del proyecto

In [1]:
import sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

# Se busca el módulo compartido en el directorio actual, el directorio padre
# y las ubicaciones habituales al trabajar con este repositorio.
PROJECT_NAME = "Machine-Learning-Project"
REPO_URL = "https://github.com/sofkp/Machine-Learning-Project.git"

candidatos = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / PROJECT_NAME,
    Path.cwd().parent / PROJECT_NAME,
    Path("/content") / PROJECT_NAME,
]

ROOT = next(
    (ruta.resolve() for ruta in candidatos if (ruta / "preprocessing.py").is_file()),
    None,
)

# En Colab, si aún no se encuentra el archivo, intentar clonar el repositorio.
# Este paso funcionará cuando preprocessing.py ya esté publicado en GitHub.
if ROOT is None and "google.colab" in sys.modules:
    destino = Path("/content") / PROJECT_NAME
    if not destino.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(destino)], check=True)
    if (destino / "preprocessing.py").is_file():
        ROOT = destino.resolve()

if ROOT is None:
    raise FileNotFoundError(
        "No se encontró preprocessing.py. Clona el repositorio con este archivo "
        "en su raíz, o súbelo a la carpeta de trabajo de Colab y vuelve a ejecutar."
    )

sys.path.insert(0, str(ROOT))
from preprocessing import load_data, split_data

# Esta carpeta será persistente si ROOT pertenece a un repositorio local;
# en Colab, los archivos deben descargarse o subirse/guardarse en GitHub.
RESULTS_DIR = ROOT / "resultados"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Carpeta del proyecto:", ROOT)
print("Carpeta de resultados:", RESULTS_DIR)
print("Versiones: scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

Carpeta del proyecto: /mnt/c/Users/sofia/Desktop/Documentos/2026/2026-2/machine/Machine-Learning-Project
Carpeta de resultados: /mnt/c/Users/sofia/Desktop/Documentos/2026/2026-2/machine/Machine-Learning-Project/resultados
Versiones: scikit-learn 1.7.2 | pandas 2.3.3 | numpy 2.2.6


## 2. Diseño experimental (protocolo compartido)

### Particiones y validación

Se emplea una única división **estratificada 60/20/20** mediante `split_data()` de `preprocessing.py`, con `random_state=42`:

- **Train (60 %, 4225 clientes):** ajustar los transformadores del preprocesamiento y entrenar los modelos.
- **Validation (20 %, 1409 clientes):** comparar configuraciones y seleccionar hiperparámetros.
- **Test (20 %, 1409 clientes):** evaluar los modelos finales una sola vez, al terminar las decisiones anteriores.

Se utiliza **validación fija (hold-out)**, no validación cruzada K-Fold. Esta decisión simplifica la coordinación y permite reproducir las mismas particiones, aunque los resultados dependen de la partición elegida.

### Modelos

1. **Dummy Classifier**, `strategy="most_frequent"`: baseline que predice la clase más frecuente aprendida de `y_train`.
2. **Regresión logística**: clasificador supervisado lineal.
3. **Árbol de decisión**: clasificador supervisado no lineal basado en reglas de partición.

El Dummy **no requiere preprocesamiento**, porque no utiliza los valores de las características para predecir. En los modelos supervisados, el preprocesamiento se integra dentro de un `Pipeline` y se ajusta exclusivamente con Train.

### Métricas y umbral

Se reportarán **Accuracy, Precision, Recall, F1-score y ROC-AUC**, más la matriz de confusión. La clase positiva es `Churn=1` (`pos_label=1`); se utiliza `zero_division=0` cuando corresponde.

El **F1-score de la clase positiva** es el criterio principal para seleccionar configuraciones: considera simultáneamente la capacidad de detectar abandonos (Recall) y la proporción de alertas correctas (Precision). Accuracy por sí solo puede resultar engañoso debido al desbalance: una predicción constante de la clase mayoritaria alcanza alrededor del 73,5 % de aciertos sin detectar abandonos. No se conocen los costos económicos de ambos tipos de error, por lo que F1 es un criterio experimental, no una estimación de costos.

Se utiliza un **umbral fijo de 0,5** para las predicciones de los modelos supervisados (`predict()`); no se optimiza el umbral mediante Validation. El ROC-AUC se calcula a partir de las **probabilidades de la clase 1**, no de las predicciones binarias.

### Selección de hiperparámetros

- Cada configuración se **entrena con Train** y se **evalúa con Validation**. Se utiliza una búsqueda explícita de configuraciones (por ejemplo, un bucle), **sin `GridSearchCV(cv=5)`**.
- Se elige la configuración con mayor **F1-score de `Churn=1` en Validation**.
- Se incluye `class_weight` ∈ `{None, "balanced"}` como una de las opciones de búsqueda para ambos modelos. No se utilizan SMOTE ni otros remuestreos.
- **Desempate:** si hay el mismo F1, se prefiere la configuración más sencilla: para el árbol, menor `max_depth` (considerando `None` sin límite) y, si persiste, mayor `min_samples_leaf`; para regresión logística, menor `C` (mayor regularización). Si persiste el empate, se conserva la primera configuración según el orden de la grilla, documentado previamente.
- Cada integrante debe conservar la grilla y la tabla de resultados obtenidos en Validation, y establecer `random_state=42` en los estimadores que lo admitan.

### Evaluación final

Después de seleccionar las configuraciones, se conserva el modelo ajustado con **Train**, sin volver a entrenar con Train + Validation. **Test** se mantiene sin utilizar durante las decisiones anteriores.

Al finalizar el trabajo de los demás integrantes, el **Dummy Classifier**, la **regresión logística** y el **árbol de decisión** seleccionados se evaluarán sobre **el mismo Test** con las mismas cinco métricas y sus matrices de confusión. Se distinguirán explícitamente los resultados de **Validation** y **Test**. No se modificarán configuraciones después de examinar Test.

### Limitaciones

Las diferencias pequeñas entre modelos pueden depender de la partición única de datos. Además, la ausencia de información temporal sobre cuándo se observaron las características limita las afirmaciones sobre predicción prospectiva: aunque se evita la fuga durante el ajuste de transformadores y modelos, no puede descartarse completamente una fuga temporal en el propio dataset.

## 3. Cargar datos y recuperar las particiones

In [2]:
# Prioridad al CSV almacenado en el repositorio, evitando depender
# de una descarga externa cuando los archivos están disponibles localmente.
DATA_PATH = ROOT / "data" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"

if DATA_PATH.is_file():
    print("Origen del dataset: CSV local", DATA_PATH)
    X, y = load_data(DATA_PATH)
else:
    print("CSV local no encontrado; se utilizará la URL pública definida en preprocessing.py")
    X, y = load_data()

X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nProporciones de clases en Train:")
print(y_train.value_counts(normalize=True).sort_index().round(4))
print("\nProporciones de clases en Validation:")
print(y_val.value_counts(normalize=True).sort_index().round(4))

# La existencia/tamaño de Test se verifica aquí, pero no se generan
# predicciones ni métricas sobre ese conjunto en esta etapa.

Origen del dataset: CSV local /mnt/c/Users/sofia/Desktop/Documentos/2026/2026-2/machine/Machine-Learning-Project/data/WA_Fn-UseC_-Telco-Customer-Churn.csv
Train: (4225, 19)
Validation: (1409, 19)
Test: (1409, 19)

Proporciones de clases en Train:
Churn
0    0.7347
1    0.2653
Name: proportion, dtype: float64

Proporciones de clases en Validation:
Churn
0    0.7346
1    0.2654
Name: proportion, dtype: float64


## 4. Entrenar el Dummy Classifier y predecir en Validation

In [3]:
baseline = DummyClassifier(strategy="most_frequent")

# El baseline aprende la clase mayoritaria únicamente a partir de Train.
baseline.fit(X_train, y_train)

y_val_pred = baseline.predict(X_val)
y_val_proba = baseline.predict_proba(X_val)[:, 1]

print("Clase mayoritaria aprendida:", baseline.classes_[np.argmax(baseline.class_prior_)])
print("Cantidad de predicciones por clase:")
print(pd.Series(y_val_pred).value_counts().sort_index())
print("Puntuaciones distintas para Churn=1:", np.unique(y_val_proba))

Clase mayoritaria aprendida: 0
Cantidad de predicciones por clase:
0    1409
Name: count, dtype: int64
Puntuaciones distintas para Churn=1: [0.]


## 5. Definir una función de evaluación común

In [4]:
def calcular_metricas(y_real, y_pred, y_proba):
    """Calcula cinco métricas para la clase positiva Churn=1."""
    return {
        "Accuracy": accuracy_score(y_real, y_pred),
        "Precision": precision_score(y_real, y_pred, pos_label=1, zero_division=0),
        "Recall": recall_score(y_real, y_pred, pos_label=1, zero_division=0),
        "F1-score": f1_score(y_real, y_pred, pos_label=1, zero_division=0),
        "ROC-AUC": roc_auc_score(y_real, y_proba),
    }

print("Función de métricas definida correctamente.")

Función de métricas definida correctamente.


## 6. Evaluar el baseline sobre Validation

In [5]:
metricas_baseline = calcular_metricas(y_val, y_val_pred, y_val_proba)

tabla_baseline = pd.DataFrame([metricas_baseline], index=["Dummy Classifier"])
tabla_baseline.index.name = "Modelo"
tabla_baseline.insert(0, "Conjunto", "Validation")

matriz = confusion_matrix(y_val, y_val_pred, labels=[0, 1])
matriz_baseline = pd.DataFrame(
    matriz,
    index=["Real No (0)", "Real Sí (1)"],
    columns=["Predicho No (0)", "Predicho Sí (1)"],
)
matriz_baseline.index.name = "Clase real"

print("Métricas en Validation:")
display(tabla_baseline.round(4))
print("Matriz de confusión en Validation (filas = clase real; columnas = predicción):")
display(matriz_baseline)

Métricas en Validation:


,Conjunto,Accuracy,Precision,Recall,F1-score,ROC-AUC
Modelo,,,,,,
Dummy Classifier,Validation,0.7346,0.0,0.0,0.0,0.5


Matriz de confusión en Validation (filas = clase real; columnas = predicción):


,Predicho No (0),Predicho Sí (1)
Clase real,,
Real No (0),1035,0
Real Sí (1),374,0


### Interpretación del baseline

En el conjunto de validacion, el Dummy obtuvo **Accuracy ≈ 0,7346**, **Precision = Recall = F1 = 0** y **ROC-AUC = 0,5**, con matriz de confusión `[[1035, 0], [374, 0]]` sobre Validation. Estos resultados deben verificarse al ejecutar este notebook con el mismo dataset y las mismas particiones.

`most_frequent` predice siempre la clase mayoritaria y asigna **probabilidad 0** a `Churn=1` en este conjunto. Su ROC-AUC es 0,5 porque las puntuaciones son constantes, no porque estime el porcentaje de abandono. El F1=0 del Dummy es una referencia mínima; superarlo no demuestra por sí solo un desempeño satisfactorio.

## 7. Exportar métricas y matriz de confusión a CSV

In [6]:
ruta_metricas = RESULTS_DIR / "resultados_baseline_validacion.csv"
ruta_matriz = RESULTS_DIR / "matriz_confusion_baseline_validacion.csv"

tabla_baseline.to_csv(ruta_metricas, encoding="utf-8-sig")
matriz_baseline.to_csv(ruta_matriz, encoding="utf-8-sig")

print("Archivos creados:")
print(" -", ruta_metricas)
print(" -", ruta_matriz)
print("\nTabla exportada:")
display(tabla_baseline)
print("Matriz exportada:")
display(matriz_baseline)

Archivos creados:
 - /mnt/c/Users/sofia/Desktop/Documentos/2026/2026-2/machine/Machine-Learning-Project/resultados/resultados_baseline_validacion.csv
 - /mnt/c/Users/sofia/Desktop/Documentos/2026/2026-2/machine/Machine-Learning-Project/resultados/matriz_confusion_baseline_validacion.csv

Tabla exportada:


,Conjunto,Accuracy,Precision,Recall,F1-score,ROC-AUC
Modelo,,,,,,
Dummy Classifier,Validation,0.734564,0.0,0.0,0.0,0.5


Matriz exportada:


,Predicho No (0),Predicho Sí (1)
Clase real,,
Real No (0),1035,0
Real Sí (1),374,0


## 8. Qué falta para cerrar la comparación final

1. Los integrantes encargados de regresión logística y árbol de decisión deben aplicar **exactamente** `load_data()`, `split_data()` y los preprocesadores del módulo compartido dentro de sus propios `Pipeline`.
2. Cada uno debe documentar las configuraciones exploradas y seleccionar sus hiperparámetros sobre **Validation**, utilizando F1 para `Churn=1`, sin ajustar umbrales.
3. Cuando todas las configuraciones estén cerradas, se evaluarán **los tres modelos**, incluido el Dummy, **una sola vez en Test** y se construirá la tabla comparativa final.
4. Publicar este notebook, `preprocessing.py`, el CSV original y el archivo de dependencias `requirements.txt` en el repositorio; documentar su ejecución en `README.md`.

